In [ ]:
import argparse
import requests
import json


In [ ]:
API_KEY = '24a58d9f2f2e9a780e9459315f20f7ca'

In [ ]:
SPORT = 'basketball_nba'
#SPORT = 'baseball_mlb'

# Bookmaker regions
# uk | us | us2 | eu | au. Multiple can be specified if comma delimited.
# More info at https://the-odds-api.com/sports-odds-data/bookmaker-apis.html
REGIONS = 'us'

# Odds markets
# More info at https://the-odds-api.com/sports-odds-data/betting-markets.html
# Note only featured markets (h2h, spreads, totals) are available with the odds endpoint.
MARKETS = 'h2h,spreads,totals'

# Odds format
# decimal | american
ODDS_FORMAT = 'american'

# Date format
# iso | unix
DATE_FORMAT = 'iso'

In [ ]:
# First get a list of events
events_response = requests.get(f'https://api.the-odds-api.com/v4/sports/{SPORT}/events', params={
    'api_key': API_KEY,
})


if events_response.status_code != 200:
    print(f'Failed to get sports: status_code {events_response.status_code}, response body {events_response.text}')
    exit()

events_json = events_response.json()
if len(events_json) == 0:
    print('No events found')
    exit()


print(f'Found {len(events_json)} events. Querying the first event')
first_event = events_json[0]
first_event_id = first_event['id']


In [ ]:

odds_response = requests.get(f'https://api.the-odds-api.com/v4/sports/{SPORT}/events/{first_event_id}/odds', params={
    'api_key': API_KEY,
    'regions': REGIONS,
    'markets': MARKETS,
    'oddsFormat': ODDS_FORMAT,
    'dateFormat': DATE_FORMAT,
})

if odds_response.status_code != 200:
    print(f'Failed to get odds: status_code {odds_response.status_code}, response body {odds_response.text}')

else:
    odds_json = odds_response.json()
    # pretty print odds response
    print(json.dumps(odds_json, indent=2))
    
    # Check the usage quota
    print('Total credits remaining', odds_response.headers['x-requests-remaining'])
    print('Total credits used', odds_response.headers['x-requests-used'])


In [10]:
import pandas as pd

print("type:", type(odds_json))
if isinstance(odds_json, dict):
    print("top-level keys:", list(odds_json.keys()))
if isinstance(odds_json, list):
    print("events count:", len(odds_json))

type: <class 'dict'>
top-level keys: ['id', 'sport_key', 'sport_title', 'commence_time', 'home_team', 'away_team', 'bookmakers']


In [11]:
import pandas as pd

events = [odds_json] if isinstance(odds_json, dict) else odds_json

rows = []
for event in events:
    event_meta = {
        "event_id": event.get("id"),
        "sport_key": event.get("sport_key"),
        "sport_title": event.get("sport_title"),
        "commence_time": event.get("commence_time"),
        "home_team": event.get("home_team"),
        "away_team": event.get("away_team"),
    }
    for bookmaker in event.get("bookmakers", []):
        book_meta = {
            "bookmaker_key": bookmaker.get("key"),
            "bookmaker_title": bookmaker.get("title"),
        }
        for market in bookmaker.get("markets", []):
            market_meta = {
                "market_key": market.get("key"),
                "market_last_update": market.get("last_update"),
            }
            for outcome in market.get("outcomes", []):
                rows.append({
                    **event_meta,
                    **book_meta,
                    **market_meta,
                    "outcome_name": outcome.get("name"),
                    "outcome_price": outcome.get("price"),
                    "outcome_point": outcome.get("point"),
                })

odds_df = pd.DataFrame(rows)
print(odds_df.head())

                           event_id       sport_key sport_title  \
0  1aae688472781f1a1aaf3efdb38e884b  basketball_nba         NBA   
1  1aae688472781f1a1aaf3efdb38e884b  basketball_nba         NBA   
2  1aae688472781f1a1aaf3efdb38e884b  basketball_nba         NBA   
3  1aae688472781f1a1aaf3efdb38e884b  basketball_nba         NBA   
4  1aae688472781f1a1aaf3efdb38e884b  basketball_nba         NBA   

          commence_time          home_team        away_team bookmaker_key  \
0  2026-06-04T00:30:00Z  San Antonio Spurs  New York Knicks       fanduel   
1  2026-06-04T00:30:00Z  San Antonio Spurs  New York Knicks       fanduel   
2  2026-06-04T00:30:00Z  San Antonio Spurs  New York Knicks       fanduel   
3  2026-06-04T00:30:00Z  San Antonio Spurs  New York Knicks       fanduel   
4  2026-06-04T00:30:00Z  San Antonio Spurs  New York Knicks       fanduel   

  bookmaker_title market_key    market_last_update       outcome_name  \
0         FanDuel        h2h  2026-06-03T19:18:11Z    New Yor